# บทที่ 1 บทนำและภูมิทัศน์ข้อมูลฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch01_intro/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch01_intro/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch01_intro/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch01_intro'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'astropy': 'astropy>=5.3', 'awkward': 'awkward>=2.4', 'h5py': 'h5py>=3.8', 'matplotlib': 'matplotlib', 'numpy': 'numpy', 'uproot': 'uproot>=5.0'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_make_sample_files.py` — โค้ดที่ 1.1, 1.2, 1.3, 1.4

Ch.1  Create small SYNTHETIC sample files in the four formats of Section 1.2.2.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_make_sample_files.py  จากโฟลเดอร์ของบท
__file__ = '01_make_sample_files.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.1  Create small SYNTHETIC sample files in the four formats of Section 1.2.2.

Book references
    ssec:example, code:root-tree, code:ligo-hdf5, code:fits-header, code:matproj-json

The files mimic the *structure* of the real open datasets (CMS dimuon ROOT tree,
LIGO strain HDF5, SDSS FITS image/spectrum, Materials Project JSON) but the
numbers are generated here. They are small (a few MB) so every reader gets the
same result without downloading gigabytes. To open the real data, see optional/.

Output (in data/)
    dimuon_sample.root                     ROOT tree "Events" with jagged muon branches
    H-H1_SYNTH_4_V1-1126259446-32.hdf5     32 s of strain at 4096 Hz
    sdss_like_image_r.fits                 256 x 256 galaxy image, r band
    sdss_like_spectrum.fits                1D stellar spectrum (loglam, flux, ivar)
    mp_like_mp-149.json                    Materials-Project-style record for Si
"""
import json
from pathlib import Path

import awkward as ak
import h5py
import numpy as np
import uproot
from astropy.io import fits

SEED = 2026
HERE = Path(__file__).resolve().parent
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)

MUON_MASS = 0.1056584  # GeV/c^2
Z_MASS, Z_WIDTH = 91.19, 2.50  # GeV/c^2


# ---------------------------------------------------------------------------
# 1) ROOT: event-level data with a variable number of muons per event
# ---------------------------------------------------------------------------
def z_pair(n):
    """Two muons whose invariant mass follows a Breit-Wigner around the Z."""
    pts, etas, phis = [], [], []
    while len(pts) < n:
        m = Z_MASS + Z_WIDTH / 2 * np.tan(np.pi * (rng.random() - 0.5))
        m += rng.normal(0, 1.5)                       # detector resolution
        if not 60 < m < 120:
            continue
        eta = rng.uniform(-2.4, 2.4, 2)
        phi = rng.uniform(-np.pi, np.pi, 2)
        pt1 = rng.uniform(25, 60)
        # massless approximation: m^2 = 2 pt1 pt2 (cosh(d_eta) - cos(d_phi))
        pt2 = m**2 / (2 * pt1 * (np.cosh(eta[0] - eta[1]) - np.cos(phi[0] - phi[1])))
        if not 10 < pt2 < 150:
            continue
        pts.append([pt1, pt2]); etas.append(list(eta)); phis.append(list(phi))
    return pts, etas, phis


def make_root(n_events=5000, frac_signal=0.6):
    n_sig = int(frac_signal * n_events)
    pt, eta, phi, charge = [], [], [], []

    s_pt, s_eta, s_phi = z_pair(n_sig)
    for i in range(n_sig):
        q = rng.choice([-1, 1])
        p, e, f, c = s_pt[i], s_eta[i], s_phi[i], [q, -q]   # opposite charge
        # about 10 % of events carry one extra soft muon
        if rng.random() < 0.10:
            p = p + [rng.exponential(8) + 3]
            e = e + [rng.uniform(-2.4, 2.4)]
            f = f + [rng.uniform(-np.pi, np.pi)]
            c = c + [int(rng.choice([-1, 1]))]
        pt.append(p); eta.append(e); phi.append(f); charge.append(c)

    for _ in range(n_events - n_sig):                     # continuum background
        k = rng.choice([1, 2, 3], p=[0.2, 0.7, 0.1])
        pt.append(list(rng.exponential(12, k) + 3))
        eta.append(list(rng.uniform(-2.4, 2.4, k)))
        phi.append(list(rng.uniform(-np.pi, np.pi, k)))
        charge.append([int(x) for x in rng.choice([-1, 1], k)])

    order = rng.permutation(n_events)                     # mix signal and background
    pick = lambda a: [a[i] for i in order]
    f32 = lambda a: ak.values_astype(ak.Array(a), np.float32)   # real file uses float
    muon = ak.zip({
        "pt": f32(pick(pt)),
        "eta": f32(pick(eta)),
        "phi": f32(pick(phi)),
        "mass": ak.full_like(f32(pick(pt)), MUON_MASS),
        "charge": ak.values_astype(ak.Array(pick(charge)), np.int32),
    })

    path = DATA / "dimuon_sample.root"
    with uproot.recreate(path) as f:
        f.mktree("Events", {"Muon": muon.type.content},
                 counter_name=lambda c: "n" + c,
                 field_name=lambda outer, inner: f"{outer}_{inner}",
                 title="SYNTHETIC dimuon events (structure of CMS open data)")
        f["Events"].extend({"Muon": muon})
    return path


# ---------------------------------------------------------------------------
# 2) HDF5: 32 s strain time series at 4096 Hz with meta and quality groups
# ---------------------------------------------------------------------------
def make_hdf5():
    fs, duration, gps0 = 4096, 32, 1126259446
    n = fs * duration
    t = np.arange(n) / fs
    strain = rng.normal(0, 1e-21, n)                      # white noise, toy level

    # toy "chirp": frequency sweeps 35 -> 250 Hz during 0.2 s before t = 16.4 s
    tc = 16.4
    m = (t > tc - 0.2) & (t < tc)
    tau = tc - t[m]
    f_inst = 35 + (250 - 35) * (1 - tau / 0.2) ** 3
    phase = 2 * np.pi * np.cumsum(f_inst) / fs
    strain[m] += 1.0e-21 * (1 - tau / 0.2) * np.sin(phase)

    path = DATA / f"H-H1_SYNTH_4_V1-{gps0}-{duration}.hdf5"
    # Same layout as a real GWOSC/LOSC file (checked against
    # H-H1_LOSC_4_V1-1126256640-4096.hdf5). Note: there is no "SampleRate"
    # entry in meta/; the sampling interval is the attribute Xspacing of Strain.
    with h5py.File(path, "w") as h:
        meta = h.create_group("meta")
        meta["Description"] = b"SYNTHETIC strain for teaching (layout of a GWOSC file)"
        meta["DescriptionURL"] = b"https://gwosc.org/"
        meta["Detector"] = b"H1"
        meta["Duration"] = duration
        meta["GPSstart"] = gps0
        meta["Observatory"] = b"H"
        meta["Type"] = b"StrainTimeSeries"
        meta["UTCstart"] = b"2015-09-14T09:50:29"
        ds = h.create_group("strain").create_dataset("Strain", data=strain)
        ds.attrs.update({"Npoints": n, "Xlabel": "GPS time", "Xspacing": 1 / fs,
                         "Xstart": gps0, "Xunits": "second", "Ylabel": "Strain",
                         "Yunits": ""})
        q = h.create_group("quality")
        simple = q.create_group("simple")
        simple["DQShortnames"] = np.array([b"DATA", b"CBC_CAT1", b"CBC_CAT2", b"CBC_CAT3",
                                           b"BURST_CAT1", b"BURST_CAT2", b"BURST_CAT3"])
        simple["DQmask"] = np.full(duration, 127, dtype=np.uint32)   # all 7 bits = good
        inj = q.create_group("injections")
        inj["InjShortnames"] = np.array([b"NO_CBC_HW_INJ", b"NO_BURST_HW_INJ",
                                         b"NO_DETCHAR_HW_INJ", b"NO_CW_HW_INJ",
                                         b"NO_STOCH_HW_INJ"])
        inj["Injmask"] = np.full(duration, 31, dtype=np.uint32)      # 1 = no injection
    return path


# ---------------------------------------------------------------------------
# 3) FITS image: header (metadata) + 2D pixel array
# ---------------------------------------------------------------------------
def make_fits_image():
    ny = nx = 256
    y, x = np.mgrid[0:ny, 0:nx]
    x0, y0, q, pa = 128.0, 128.0, 0.6, np.deg2rad(30)
    xr = (x - x0) * np.cos(pa) + (y - y0) * np.sin(pa)
    yr = -(x - x0) * np.sin(pa) + (y - y0) * np.cos(pa)
    r = np.hypot(xr, yr / q)
    galaxy = 40.0 * np.exp(-r / 12.0)                     # exponential disk
    sky = 5.0
    img = rng.poisson(galaxy + sky).astype(np.float32) - sky

    hdr = fits.Header()
    hdr["RA"] = (179.689453125, "Right Ascension [deg]")
    hdr["DEC"] = (0.28563112068, "Declination [deg]")
    hdr["FILTER"] = ("r", "r-band")
    hdr["EXPTIME"] = (53.91, "exposure time [s]")
    hdr["BUNIT"] = ("counts", "pixel unit (sky subtracted)")
    hdr["COMMENT"] = "SYNTHETIC image for teaching; header keywords follow SDSS style"
    path = DATA / "sdss_like_image_r.fits"
    fits.PrimaryHDU(data=img, header=hdr).writeto(path, overwrite=True)
    return path


# ---------------------------------------------------------------------------
# 4) FITS spectrum: binary table with log10(wavelength), flux, inverse variance
# ---------------------------------------------------------------------------
def make_fits_spectrum():
    loglam = np.arange(np.log10(3800), np.log10(9200), 1e-4)   # SDSS-like sampling
    lam = 10**loglam                                            # Angstrom
    h, c, k, T = 6.626e-34, 2.998e8, 1.381e-23, 6000.0
    lam_m = lam * 1e-10
    bb = 1 / lam_m**5 / (np.exp(h * c / (lam_m * k * T)) - 1)
    flux = 50 * bb / bb.max()
    for line, depth, width in [(6563, 0.35, 6), (4861, 0.30, 5), (4340, 0.25, 5),
                               (5892, 0.20, 3)]:                # H-alpha/beta/gamma, Na D
        flux *= 1 - depth * np.exp(-0.5 * ((lam - line) / width) ** 2)
    sigma = 0.02 * flux.max() * np.ones_like(flux)
    flux_obs = flux + rng.normal(0, sigma)

    cols = fits.ColDefs([
        fits.Column(name="loglam", format="E", array=loglam),
        fits.Column(name="flux", format="E", array=flux_obs, unit="arbitrary"),
        fits.Column(name="ivar", format="E", array=1 / sigma**2),
    ])
    hdr = fits.Header()
    hdr["OBJTYPE"] = ("STAR", "object class")
    hdr["TEFF"] = (T, "temperature used to generate the continuum [K]")
    hdr["COMMENT"] = "SYNTHETIC spectrum for teaching (blackbody + 4 absorption lines)"
    path = DATA / "sdss_like_spectrum.fits"
    fits.HDUList([fits.PrimaryHDU(header=hdr),
                  fits.BinTableHDU.from_columns(cols, name="COADD")]).writeto(path, overwrite=True)
    return path


# ---------------------------------------------------------------------------
# 5) JSON: nested key-value record, as returned by the Materials Project API
# ---------------------------------------------------------------------------
def make_json():
    rec = {
        "_note": "Structure of an MP summary record. Fields set to null must be "
                 "fetched from the API (see optional/fetch_real_data.py).",
        "material_id": "mp-149",
        "formula_pretty": "Si",
        "symmetry": {"crystal_system": "Cubic", "symbol": "Fd-3m", "number": 227},
        "formation_energy_per_atom": 0.0,   # elemental reference, zero by definition
        "energy_above_hull": 0.0,           # ground state
        "is_stable": True,
        "band_gap": None,                   # eV, from DFT -> fetch from API
        "density": None,                    # g/cm^3       -> fetch from API
    }
    path = DATA / "mp_like_mp-149.json"
    path.write_text(json.dumps(rec, indent=2, ensure_ascii=False), encoding="utf-8")
    return path


if __name__ == "__main__":
    for fn in (make_root, make_hdf5, make_fits_image, make_fits_spectrum, make_json):
        p = fn()
        print(f"{p.name:42s} {p.stat().st_size / 1024:9.1f} kB")

## `02_inspect_root.py` — โค้ดที่ 1.1

Ch.1  Open the ROOT tree and see why event-level data is 'jagged'.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_inspect_root.py  จากโฟลเดอร์ของบท
__file__ = '02_inspect_root.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.1  Open the ROOT tree and see why event-level data is 'jagged'.

Book references
    sssec:cerndata, code:root-tree

What this script shows
    1. the branch list of the tree (compare with code:root-tree)
    2. the number of muons differs from event to event
    3. a jagged array cannot be a plain rectangular table without a choice
       (pad, truncate, or explode to one row per muon)
    4. a first physics look: the dimuon invariant mass shows the Z peak
"""
from pathlib import Path

import awkward as ak
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import uproot

HERE = Path(__file__).resolve().parent
DATA, OUT = HERE / "data", HERE / "outputs"
OUT.mkdir(exist_ok=True)

tree = uproot.open(DATA / "dimuon_sample.root")["Events"]

# 1) structure ---------------------------------------------------------------
print(f"Tree 'Events': {tree.num_entries} events")
for name, br in tree.items():
    print(f"  {name:12s} {br.typename}")

ev = tree.arrays(library="ak")

# 2) variable length ---------------------------------------------------------
counts = np.bincount(ak.to_numpy(ev.nMuon))
print("\nnMuon  number_of_events")
for k, c in enumerate(counts):
    if c:
        print(f"  {k:3d}   {c}")
print("first 3 events, Muon_pt:", ak.to_list(ev.Muon_pt[:3]))

# 3) three ways to force a rectangular table ---------------------------------
padded = ak.to_numpy(ak.fill_none(ak.pad_none(ev.Muon_pt, 2, clip=True), np.nan))
print("\n(a) pad/clip to 2 muons -> shape", padded.shape,
      "| events that lose muons:", int(ak.sum(ev.nMuon > 2)))
print("(b) keep only events with nMuon == 2 ->", int(ak.sum(ev.nMuon == 2)), "rows")
print("(c) explode to one row per muon ->", int(ak.sum(ev.nMuon)), "rows")

# 4) invariant mass of the two leading muons, opposite charge ---------------
sel = ev[(ev.nMuon >= 2)]
idx = ak.argsort(sel.Muon_pt, ascending=False)
pt, eta, phi, q = (sel[b][idx][:, :2] for b in ("Muon_pt", "Muon_eta", "Muon_phi", "Muon_charge"))
os_mask = (q[:, 0] * q[:, 1]) < 0
m2 = 2 * pt[:, 0] * pt[:, 1] * (np.cosh(eta[:, 0] - eta[:, 1]) - np.cos(phi[:, 0] - phi[:, 1]))
mass = ak.to_numpy(np.sqrt(m2[os_mask]))
in_window = (mass > 81) & (mass < 101)
print(f"\nopposite-charge pairs: {len(mass)}; with 81 < m < 101 GeV: {in_window.sum()}")
print(f"median mass in window: {np.median(mass[in_window]):.2f} GeV/c^2 (Z = 91.19)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(mass, bins=np.linspace(0, 150, 76), histtype="step", color="k")
ax.set_xlabel(r"dimuon invariant mass $m_{\mu\mu}$ [GeV/$c^2$]")
ax.set_ylabel("events / 2 GeV")
ax.set_title("SYNTHETIC CMS-like dimuon sample")
fig.tight_layout()
fig.savefig(OUT / "ch01_dimuon_mass.png", dpi=150)
print("saved outputs/ch01_dimuon_mass.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `03_inspect_hdf5.py` — โค้ดที่ 1.2

Ch.1  Walk an HDF5 file (groups, datasets, attributes) and check the numbers

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_inspect_hdf5.py  จากโฟลเดอร์ของบท
__file__ = '03_inspect_hdf5.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.1  Walk an HDF5 file (groups, datasets, attributes) and check the numbers
quoted in Section 1.2.2 for the LIGO strain file.

Book references
    sssec:ligo, code:ligo-hdf5, misconception box "strain 1e-21 is a distance"

Checks
    sampling rate comes from the Strain attribute Xspacing (no SampleRate in meta/)
    32 s x 4096 Hz = 131,072 samples
    dL = h * L with h ~ 1e-21 and L = 4 km  ->  4e-18 m, a few hundred times
    smaller than a proton (~1e-15 m)
"""
from pathlib import Path

import h5py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

HERE = Path(__file__).resolve().parent
DATA, OUT = HERE / "data", HERE / "outputs"
OUT.mkdir(exist_ok=True)
path = next(DATA.glob("H-H1_*.hdf5"))


def show(name, obj):
    depth = name.count("/")
    pad = "  " * depth
    if isinstance(obj, h5py.Group):
        print(f"{pad}{name.split('/')[-1]}/")
    else:
        val = obj[()] if obj.shape == () else f"array shape={obj.shape} dtype={obj.dtype}"
        if isinstance(val, bytes):
            val = val.decode()
        print(f"{pad}{name.split('/')[-1]:14s} {val}")
    for k, v in obj.attrs.items():
        print(f"{pad}   @{k} = {v}")


with h5py.File(path, "r") as h:
    print(path.name)
    h.visititems(show)
    # the sampling rate is NOT in meta/: read it from the Strain attribute
    dt = h["strain/Strain"].attrs["Xspacing"]
    fs = int(round(1 / dt))
    duration = int(h["meta/Duration"][()])
    strain = h["strain/Strain"][:]
    dq = h["quality/simple/DQmask"][()]

n = len(strain)
print(f"\nsamples = {n:,}  (expected {duration} s x {fs} Hz = {duration * fs:,})")
assert n == duration * fs
print(f"sampling interval from attribute Xspacing = {dt} s -> {fs} Hz")
print(f"DQmask: {len(dq)} values (one per second), all == 127 (7 flags passed): {bool((dq == 127).all())}")

# The misconception box: strain is a ratio; only dL = h L is a length
h_peak, L = 1e-21, 4e3
dL = h_peak * L
proton = 1e-15
print(f"dL = h L = {h_peak:.0e} x {L:.0f} m = {dL:.1e} m")
print(f"proton diameter / dL = {proton / dL:.0f}  -> 'several hundred times smaller'")

t = np.arange(n) / fs
fig, ax = plt.subplots(2, 1, figsize=(7, 5))
ax[0].plot(t, strain, lw=0.3, color="k")
ax[0].set_xlabel("time since GPS start [s]"); ax[0].set_ylabel("strain h")
ax[0].set_title("SYNTHETIC strain, 32 s")
m = (t > 16.0) & (t < 16.5)
ax[1].plot(t[m], strain[m], lw=0.6, color="k")
ax[1].set_xlabel("time [s]"); ax[1].set_ylabel("strain h")
ax[1].set_title("zoom: toy chirp buried in noise")
fig.tight_layout()
fig.savefig(OUT / "ch01_strain.png", dpi=150)
print("saved outputs/ch01_strain.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `04_inspect_fits.py` — โค้ดที่ 1.3

Ch.1  Read FITS files: header keywords (metadata) and data (image or table).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_inspect_fits.py  จากโฟลเดอร์ของบท
__file__ = '04_inspect_fits.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.1  Read FITS files: header keywords (metadata) and data (image or table).

Book references
    sssec:sdss, code:fits-header, exercise ex:matching items (b) and (e)

The same file format holds two modalities here:
    sdss_like_image_r.fits   -> image    (2D array in the primary HDU)
    sdss_like_spectrum.fits  -> spectrum (binary table in extension 1)
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from astropy.io import fits

HERE = Path(__file__).resolve().parent
DATA, OUT = HERE / "data", HERE / "outputs"
OUT.mkdir(exist_ok=True)

# ---- image -----------------------------------------------------------------
with fits.open(DATA / "sdss_like_image_r.fits") as hdul:
    hdul.info()
    hdr, img = hdul[0].header, hdul[0].data

print("\nheader (compare with code:fits-header):")
for key in ("SIMPLE", "NAXIS", "NAXIS1", "NAXIS2", "RA", "DEC", "FILTER", "EXPTIME"):
    print(f"  {key:8s}= {hdr[key]!r:>20}  / {hdr.comments[key]}")

print(f"\nimage shape {img.shape}, dtype {img.dtype}")
print(f"pixel values: min {img.min():.1f}, median {np.median(img):.2f}, max {img.max():.1f}")
print(f"counts per second at the brightest pixel: {img.max() / hdr['EXPTIME']:.2f}"
      "   <- needs EXPTIME from the header")

# ---- spectrum --------------------------------------------------------------
with fits.open(DATA / "sdss_like_spectrum.fits") as hdul:
    print()
    hdul.info()
    tab = hdul["COADD"].data
lam = 10 ** tab["loglam"]
flux = tab["flux"]
print(f"\nspectrum: {len(lam)} pixels, {lam.min():.0f}-{lam.max():.0f} Angstrom")
print(f"wavelength step grows with lambda: {lam[1]-lam[0]:.3f} A at blue end, "
      f"{lam[-1]-lam[-2]:.3f} A at red end (constant step in log10)")
i_ha = np.argmin(np.abs(lam - 6563))
win = (lam > 6500) & (lam < 6620)
print(f"flux at H-alpha 6563 A is {flux[i_ha] / np.median(flux[win]):.2f} of the local median"
      " -> absorption line")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(np.arcsinh(img / 5), origin="lower", cmap="gray")
ax[0].set_title(f"SYNTHETIC r-band image  {img.shape[1]}x{img.shape[0]} px")
ax[1].plot(lam, flux, lw=0.5, color="k")
for line in (4340, 4861, 5892, 6563):
    ax[1].axvline(line, ls=":", color="gray")
ax[1].set_xlabel(r"wavelength [$\AA$]"); ax[1].set_ylabel("flux [arb.]")
ax[1].set_title("SYNTHETIC stellar spectrum")
fig.tight_layout()
fig.savefig(OUT / "ch01_fits_image_spectrum.png", dpi=150)
print("saved outputs/ch01_fits_image_spectrum.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `05_inspect_json.py` — โค้ดที่ 1.4

Ch.1  Read a nested JSON record and apply physics sanity checks to it.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_inspect_json.py  จากโฟลเดอร์ของบท
__file__ = '05_inspect_json.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.1  Read a nested JSON record and apply physics sanity checks to it.

Book references
    sssec:matproj, code:matproj-json, misconception box "database values are true values"

Checks
    formation energy of a pure element must be 0 (it is the reference state)
    energy_above_hull == 0  <=>  is_stable == True
    missing fields (null) are reported, never silently filled
"""
import json
from pathlib import Path

HERE = Path(__file__).resolve().parent
rec = json.loads((HERE / "data" / "mp_like_mp-149.json").read_text(encoding="utf-8"))


def walk(d, prefix=""):
    for k, v in d.items():
        if k.startswith("_"):
            continue
        if isinstance(v, dict):
            print(f"{prefix}{k}:")
            walk(v, prefix + "    ")
        else:
            print(f"{prefix}{k:28s} {v!r}")


walk(rec)

print("\nnested access: rec['symmetry']['number'] =", rec["symmetry"]["number"])

elements = {"Si"}
is_element = rec["formula_pretty"] in elements
print("\nsanity checks")
if is_element:
    ok = rec["formation_energy_per_atom"] == 0.0
    print(f"  pure element -> formation energy 0 : {'OK' if ok else 'FAIL'}")
ok = (rec["energy_above_hull"] == 0.0) == rec["is_stable"]
print(f"  e_above_hull == 0 <=> is_stable     : {'OK' if ok else 'FAIL'}")
missing = [k for k, v in rec.items() if v is None]
print(f"  fields still to fetch from the API  : {missing}")
print("\nreminder: band_gap from GGA-DFT is systematically lower than experiment;"
      " a model trained on it learns that bias too (veracity).")

## `06_crystal_graph.py`

Ch.1  The fifth modality: a graph. Build the neighbour graph of crystalline Si.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_crystal_graph.py  จากโฟลเดอร์ของบท
__file__ = '06_crystal_graph.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.1  The fifth modality: a graph. Build the neighbour graph of crystalline Si.

Book references
    tab:modalities (row "graph"), exercise ex:matching note on item (d)

Atoms are nodes; an edge joins two atoms closer than a cutoff. Periodic
boundary conditions are applied with the minimum-image convention.
Lattice constant: experimental a = 5.431 Angstrom (room temperature).
Physics check: in the diamond structure every Si atom has 4 nearest
neighbours at a*sqrt(3)/4.
"""
import itertools

import numpy as np

A = 5.431  # Angstrom, experimental lattice constant of Si

# 8 atoms of the conventional cubic cell of the diamond structure (fractional)
fcc = np.array([[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]])
basis = np.vstack([fcc, fcc + 0.25])
N_REP = 2                                   # 2 x 2 x 2 supercell -> 64 atoms
cells = np.array(list(itertools.product(range(N_REP), repeat=3)))
frac = (basis[None, :, :] + cells[:, None, :]).reshape(-1, 3) / N_REP
box = A * N_REP
pos = frac * box
n = len(pos)

d = pos[:, None, :] - pos[None, :, :]
d -= box * np.round(d / box)                 # minimum image
dist = np.linalg.norm(d, axis=-1)

d_nn = A * np.sqrt(3) / 4
cutoff = 0.5 * (d_nn + A / np.sqrt(2))       # halfway to the 2nd shell (a/sqrt(2))
adj = (dist > 0) & (dist < cutoff)
edges = np.argwhere(np.triu(adj))
degree = adj.sum(1)

print(f"atoms (nodes)            : {n}")
print(f"edges                    : {len(edges)}")
print(f"degree of every node     : {sorted(set(degree.tolist()))}  (diamond -> 4)")
print(f"nearest-neighbour length : {dist[adj].min():.3f} A  "
      f"(a*sqrt(3)/4 = {d_nn:.3f} A)")
print(f"edges per atom           : {len(edges) / n:.1f}  (= 4/2, each bond shared)")
print("\nfirst 5 edges (i, j, length in A):")
for i, j in edges[:5]:
    print(f"  {i:2d} {j:2d} {dist[i, j]:.3f}")

## `07_compare_formats.py`

Ch.1  One table for all sample files: format, modality, shape, size, metadata.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_compare_formats.py  จากโฟลเดอร์ของบท
__file__ = '07_compare_formats.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.1  One table for all sample files: format, modality, shape, size, metadata.

Book references
    ssec:synthesis, tab:modalities, exercise ex:matching

Run after 01-05. Writes outputs/ch01_format_summary.csv.
"""
import csv
import json
from pathlib import Path

import h5py
import uproot
from astropy.io import fits

HERE = Path(__file__).resolve().parent
DATA, OUT = HERE / "data", HERE / "outputs"
OUT.mkdir(exist_ok=True)

rows = []

p = DATA / "dimuon_sample.root"
t = uproot.open(p)["Events"]
rows.append(dict(file=p.name, format="ROOT", modality="tabular (jagged)",
                 shape=f"{t.num_entries} events x {len(t.keys())} branches",
                 metadata_items="1 (tree title)"))

p = next(DATA.glob("H-H1_*.hdf5"))
with h5py.File(p) as h:
    n_meta = len(h["meta"]) + sum(len(o.attrs) for o in
                                  [h[k] for k in ("strain/Strain", "quality/injections")])
    rows.append(dict(file=p.name, format="HDF5", modality="time series",
                     shape=f"{h['strain/Strain'].shape[0]} samples",
                     metadata_items=n_meta))

p = DATA / "sdss_like_image_r.fits"
with fits.open(p) as hd:
    rows.append(dict(file=p.name, format="FITS", modality="image",
                     shape="x".join(map(str, hd[0].data.shape)),
                     metadata_items=len(hd[0].header)))

p = DATA / "sdss_like_spectrum.fits"
with fits.open(p) as hd:
    rows.append(dict(file=p.name, format="FITS", modality="spectrum",
                     shape=f"{len(hd[1].data)} wavelengths",
                     metadata_items=len(hd[0].header) + len(hd[1].header)))

p = DATA / "mp_like_mp-149.json"
rec = json.loads(p.read_text(encoding="utf-8"))
rows.append(dict(file=p.name, format="JSON", modality="tabular (one row per material)",
                 shape=f"{sum(1 for k in rec if not k.startswith('_'))} top-level fields",
                 metadata_items="-"))

for r in rows:
    r["size_kB"] = round((DATA / r["file"]).stat().st_size / 1024, 1)

cols = ["file", "format", "modality", "shape", "size_kB", "metadata_items"]
w = [max(len(str(r[c])) for r in rows + [dict(zip(cols, cols))]) for c in cols]
print("  ".join(c.ljust(x) for c, x in zip(cols, w)))
for r in rows:
    print("  ".join(str(r[c]).ljust(x) for c, x in zip(cols, w)))

with open(OUT / "ch01_format_summary.csv", "w", newline="", encoding="utf-8") as f:
    wr = csv.DictWriter(f, fieldnames=cols)
    wr.writeheader(); wr.writerows(rows)
print("\nsaved outputs/ch01_format_summary.csv")

# ข้อมูลจริง (ไม่บังคับ)
สคริปต์สำหรับดาวน์โหลดและอ่านข้อมูลจริงอยู่ใน `ch01_intro/optional/` วิธีดาวน์โหลดและผลที่ควรได้อยู่ใน `ch01_intro/README.md` ข้อมูลเหล่านี้ไม่ได้อยู่ใน GitHub เพราะขนาดใหญ่หรือมีเงื่อนไขสัญญาอนุญาต